# E2AD: Official BR35H Baseline Reproduction
**Paper**: *Anomaly Detection in Medical Images Using Encoder-Attention-2Decoders Reconstruction* (IEEE Transactions on Medical Imaging 2025)  
**Experiment Identifier**: `E2AD-BR35H-OFFICIAL-BASELINE`  
**Personal FYP Repository**: [gnanadeep256/E2AD](https://github.com/gnanadeep256/E2AD)  
**Upstream Reference**: [TumCCC/E2AD](https://github.com/TumCCC/E2AD)  

---
## 1. Environment Verification
Verify Python runtime, PyTorch, Torchvision, Albumentations, CUDA availability, and GPU device details.


In [ ]:
import sys
import os
import torch
import torchvision
import albumentations as A

print("=" * 65)
print("SYSTEM & LIBRARY ENVIRONMENT AUDIT")
print("=" * 65)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Torchvision Version  : {torchvision.__version__}")
print(f"Albumentations Ver   : {A.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Device Count    : {torch.cuda.device_count()}")
    print(f"Active Device ID     : 0")
    print(f"Device Name          : {torch.cuda.get_device_name(0)}")
    print(f"Device Capability    : {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM (GB)      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[WARNING] CUDA is NOT available! GPU accelerator is required for E2AD training.")

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Primary Torch Device : {device}")
print("=" * 65)


## 2. Repository & Working Directory Verification
Confirm repository directory and check that all core source files exist.


In [ ]:
import os
import sys

REPO_DIR = "/kaggle/working/E2AD"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.exists("./e2ad_br35h.py"):
    REPO_DIR = os.path.abspath(".")
else:
    print(f"Cloning personal FYP repository to {REPO_DIR}...")
    os.system(f"git clone https://github.com/gnanadeep256/E2AD.git {REPO_DIR}")
    os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Current Working Directory: {os.getcwd()}")

REQUIRED_FILES = [
    "e2ad_br35h.py",
    "train_utils.py",
    "utils.py",
    "methods/edc1.py",
    "models/edc.py",
    "models/resnet.py",
    "models/resnet_decoder.py",
    "datasets/dataset.py",
    "datasets/transforms.py",
    "prepare_dataset/prepare_br35h.py"
]

missing_files = [f for f in REQUIRED_FILES if not os.path.exists(f)]
if missing_files:
    raise FileNotFoundError(f"Missing essential files: {missing_files}")
else:
    print(f"All {len(REQUIRED_FILES)} required repository source files verified successfully!")


## 3. Dataset Setup & Verification (BR35H)
The official E2AD data pipeline requires BR35H structured into:
- `train/NORMAL/`: 1,000 normal MRI slices
- `test/NORMAL/`: 500 normal MRI slices
- `test/ABNORMAL/`: 1,500 abnormal (tumor) MRI slices

This section checks for an existing preprocessed dataset or runs `prepare_br35h.py` on the raw Kaggle dataset.


In [ ]:
import os
import shutil
from PIL import Image

PROCESSED_DATA_DIR = "/kaggle/working/BR35H"

CANDIDATE_RAW_DIRS = [
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection",
    "/kaggle/input/brain-tumor-detection",
    "/kaggle/input/brain-tumor-detection/brain-tumor-detection",
    "/kaggle/input/br35h-dataset",
    "/kaggle/input/br35h",
    "./BR35H/original",
    "../BR35H/original"
]

raw_data_folder = None
for cand in CANDIDATE_RAW_DIRS:
    if os.path.exists(cand):
        subdirs = os.listdir(cand)
        if 'no' in subdirs and 'yes' in subdirs:
            raw_data_folder = cand
            break
        for sub in subdirs:
            nested = os.path.join(cand, sub)
            if os.path.isdir(nested) and 'no' in os.listdir(nested) and 'yes' in os.listdir(nested):
                raw_data_folder = nested
                break
        if raw_data_folder:
            break

print(f"Raw BR35H Data Folder Identified: {raw_data_folder}")

def verify_processed_counts(base_dir):
    try:
        tr_norm = len(os.listdir(os.path.join(base_dir, "train", "NORMAL")))
        te_norm = len(os.listdir(os.path.join(base_dir, "test", "NORMAL")))
        te_abnorm = len(os.listdir(os.path.join(base_dir, "test", "ABNORMAL")))
        return (tr_norm == 1000 and te_norm == 500 and te_abnorm == 1500), (tr_norm, te_norm, te_abnorm)
    except Exception:
        return False, (0, 0, 0)

is_complete, counts = verify_processed_counts(PROCESSED_DATA_DIR)

if not is_complete:
    print(f"Processed dataset not found or incomplete. Current counts: {counts}. Preprocessing raw data...")
    if raw_data_folder is None:
        raise FileNotFoundError(
            "Could not find raw BR35H dataset with 'no' and 'yes' folders. "
            "Please ensure the Kaggle dataset 'ahmedhamada0/brain-tumor-detection' is attached."
        )
    cmd = f"python prepare_dataset/prepare_br35h.py --data-folder '{raw_data_folder}' --save-folder '{PROCESSED_DATA_DIR}'"
    print(f"Running: {cmd}")
    res = os.system(cmd)
    if res != 0:
        raise RuntimeError("Dataset preprocessing script failed!")
    is_complete, counts = verify_processed_counts(PROCESSED_DATA_DIR)

print(f"BR35H Verification Passed:")
print(f"  - train/NORMAL   : {counts[0]} / 1000 images")
print(f"  - test/NORMAL    : {counts[1]} / 500 images")
print(f"  - test/ABNORMAL  : {counts[2]} / 1500 images")
assert is_complete, f"Counts mismatch: {counts}"

sample_tr = os.path.join(PROCESSED_DATA_DIR, "train", "NORMAL", os.listdir(os.path.join(PROCESSED_DATA_DIR, "train", "NORMAL"))[0])
sample_te_n = os.path.join(PROCESSED_DATA_DIR, "test", "NORMAL", os.listdir(os.path.join(PROCESSED_DATA_DIR, "test", "NORMAL"))[0])
sample_te_a = os.path.join(PROCESSED_DATA_DIR, "test", "ABNORMAL", os.listdir(os.path.join(PROCESSED_DATA_DIR, "test", "ABNORMAL"))[0])

for s_path in [sample_tr, sample_te_n, sample_te_a]:
    with Image.open(s_path) as img:
        img.verify()
print("Sample images verified for integrity (valid PNG/JPEG, uncorrupted).")


## 4. Kaggle Compatibility Audit Summary

The original official E2AD code was built against:
- Python 3.8.12
- PyTorch 1.12.0 + CUDA 11.3
- Torchvision 0.13.0
- Albumentations $\le 1.1$

Modern Kaggle environments (e.g. Python 3.11/3.12, PyTorch 2.x, Torchvision 0.25+, Albumentations 1.4+/2.0+) require two minimal compatibility adjustments:

1. **Torchvision Internal Deprecation**:
   `torchvision._internally_replaced_utils.load_state_dict_from_url` was removed in Torchvision $\ge 0.14$. Replaced cleanly with standard `torch.hub.load_state_dict_from_url` in `models/resnet.py` and `models/resnet_decoder.py`.
2. **Albumentations `to_tuple` Removal**:
   `to_tuple` was removed from `albumentations.core.transforms_interface` in Albumentations $\ge 1.4$. Provided via a safe fallback definition in `datasets/transforms.py`.
3. **Execution Flags**:
   - Single GPU on Kaggle must be targeted with `--gpu 0` (default in code was `'1'`).
   - Checkpoint saving must be enabled with `--save_weight 1` (default was 0).
   - Kaggle **Internet** must be **ON** to download ResNet-50 ImageNet pretrained weights.


## 5. Torchvision Compatibility Patch & Verification
Verify that `models/resnet.py` and `models/resnet_decoder.py` import `load_state_dict_from_url` from `torch.hub`.


In [ ]:
def check_or_patch_torchvision():
    target_files = ['models/resnet.py', 'models/resnet_decoder.py']
    for tf in target_files:
        if not os.path.exists(tf):
            continue
        with open(tf, 'r', encoding='utf-8') as f:
            content = f.read()
        if 'torchvision._internally_replaced_utils' in content:
            print(f"Applying patch to {tf}...")
            content = content.replace(
                'from torchvision._internally_replaced_utils import load_state_dict_from_url',
                'from torch.hub import load_state_dict_from_url'
            )
            with open(tf, 'w', encoding='utf-8') as f:
                f.write(content)
        else:
            print(f"{tf} verified: using torch.hub.load_state_dict_from_url.")

check_or_patch_torchvision()

from models.resnet import resnet50
from models.resnet_decoder import resnet50_decoder
print("ResNet models and decoders imported successfully without torchvision deprecation errors.")


## 6. Albumentations Compatibility Patch & Verification
Verify that `datasets/transforms.py` imports without error under modern Albumentations.


In [ ]:
def check_or_patch_albumentations():
    tf = 'datasets/transforms.py'
    if not os.path.exists(tf):
        return
    with open(tf, 'r', encoding='utf-8') as f:
        content = f.read()
    
    old_import = """from albumentations.core.transforms_interface import (
    DualTransform,
    ImageOnlyTransform,
    NoOp,
    to_tuple,
)"""
    new_import = """from albumentations.core.transforms_interface import (
    DualTransform,
    ImageOnlyTransform,
    NoOp,
)
try:
    from albumentations.core.transforms_interface import to_tuple
except ImportError:
    def to_tuple(param, low=None, bias=None):
        return tuple(param) if isinstance(param, (list, tuple)) else (param, param)"""

    if old_import in content:
        print(f"Patching {tf} for Albumentations compatibility...")
        content = content.replace(old_import, new_import)
        with open(tf, 'w', encoding='utf-8') as f:
            f.write(content)
    else:
        print(f"{tf} verified: includes safe to_tuple fallback.")

check_or_patch_albumentations()

from datasets.dataset import AD_Dataset
from datasets.transforms import PixelShuffle, CutMix, MeanDropout
import e2ad_br35h
print("Dataset and transform modules imported successfully!")


## 7. Official DataLoader Smoke Test
Verify batch generation, data shapes, normalization, and tensor types using the official `AD_Dataset` and `get_data_loader`.


In [ ]:
from datasets.dataset import AD_Dataset
from datasets.data_utils import get_data_loader
import torch

generator_lb = torch.Generator()
generator_lb.manual_seed(0)

raw_train_dset = AD_Dataset(name='br35h', train=True, transform=False, data_dir=f"{PROCESSED_DATA_DIR}/")
train_dset = raw_train_dset.get_dset()

raw_eval_dset = AD_Dataset(name='br35h', train=False, transform=False, data_dir=f"{PROCESSED_DATA_DIR}/")
eval_dset = raw_eval_dset.get_dset()

print(f"Verified Train Dataset Size : {len(train_dset)} samples")
print(f"Verified Eval Dataset Size  : {len(eval_dset)} samples")

train_loader = get_data_loader(
    train_dset,
    batch_size=32,
    data_sampler='RandomSampler',
    num_iters=100,
    num_workers=2,
    distributed=False,
    generator=generator_lb
)

batch = next(iter(train_loader))
idx, x, xo, y, filenames = batch

print("-" * 55)
print(f"Batch Tensor (x) Shape      : {list(x.shape)} (Expected: [32, 3, 256, 256])")
print(f"Original Tensor (xo) Shape  : {list(xo.shape)}")
print(f"Labels Tensor (y) Shape     : {list(y.shape)} (Expected: [32])")
print(f"Batch Dtype                 : {x.dtype}")
print(f"Value Range                 : min={x.min().item():.3f}, max={x.max().item():.3f}")
print("-" * 55)

assert x.shape == (32, 3, 256, 256), f"Unexpected shape {x.shape}"
assert x.dtype == torch.float32, f"Unexpected dtype {x.dtype}"
print("DataLoader smoke test PASSED successfully!")


## 8. Model Initialization & Forward Pass Smoke Test
Verify that the `E2AD` architecture instantiates, transfers to GPU, and executes a forward pass without CUDA or shape errors.


In [ ]:
import torch
from models.edc import E2AD
from utils import count_parameters

print("Instantiating official E2AD architecture (ResNet-50 encoder + dual decoders + SASC + TCCL)...")
model = E2AD(bn_pretrain=False)

trainable_params = count_parameters(model)
print(f"Total Trainable Parameters: {trainable_params:,}")

model = model.to(device)
model.eval()

dummy_input = torch.randn(2, 3, 256, 256, device=device)

with torch.no_grad():
    outputs = model(dummy_input)

print("\nForward Pass Output Keys & Tensor Shapes:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        print(f"  - {k:<15} : shape={list(v.shape)}, dtype={v.dtype}")
    else:
        print(f"  - {k:<15} : {v}")

assert 'loss' in outputs, "Model output missing 'loss' key!"
assert 'p_all_1' in outputs and 'p_all_2' in outputs, "Model output missing decoder anomaly maps!"
print("\nModel initialization and GPU forward pass PASSED successfully!")

# Explicitly cleanup smoke test model from GPU VRAM to leave full memory for training
del model, dummy_input, outputs
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU smoke test model successfully freed from VRAM.")


## 9. Baseline Training Execution
Execute the official training routine (`e2ad_br35h.py`) with:
- Model: `E2AD`
- Optimizer: `AdamW` (lr=5e-4, lr_encoder=5e-5, weight_decay=1e-4)
- Iterations: 4,000 (eval every 100 iters)
- Batch size: 32 (eval batch size: 64)
- GPU: `0`
- `--save_weight 1` to persist checkpoints to disk
- `--amp True`: Uses official Automatic Mixed Precision (`torch.cuda.amp`) built into `e2ad_br35h.py` to prevent OOM on 15 GB Tesla T4 GPUs (the authors trained on a 40 GB NVIDIA A100).


In [ ]:
import subprocess
import time
import os
import gc
import torch

# Prevent PyTorch allocator fragmentation on 16GB GPUs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Flush any lingering CUDA memory before launching training process
gc.collect()
torch.cuda.empty_cache()

start_time = time.time()

TRAIN_CMD = [
    "python", "e2ad_br35h.py",
    "--train_times", "1",
    "--gpu", "0",
    "--model_name", "E2AD",
    "--data_dir", f"{PROCESSED_DATA_DIR}/",
    "--save_weight", "1",
    "--save_dir", "./saved_models",
    "--save_name", "e2ad_br35h",
    "--num_train_iter", "4000",
    "--num_eval_iter", "100",
    "--batch_size", "32",
    "--eval_batch_size", "64",
    "--optim", "AdamW",
    "--lr", "5e-4",
    "--lr_encoder", "5e-5",
    "--weight_decay", "1e-4",
    "--amp", "True",
    "--seed", "0"
]

print("Launching Official E2AD BR35H Baseline Run:")
print("Command:", " ".join(TRAIN_CMD))
print("=" * 65)

process = subprocess.Popen(
    TRAIN_CMD,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

log_lines = []
for line in process.stdout:
    print(line, end="")
    log_lines.append(line)

process.wait()
total_train_time = time.time() - start_time

print("=" * 65)
print(f"Training completed with exit code: {process.returncode}")
print(f"Total Elapsed Time: {total_train_time / 60:.2f} minutes")

os.makedirs("reports", exist_ok=True)
with open("reports/training_br35h_baseline.log", "w", encoding="utf-8") as f:
    f.writelines(log_lines)
print("Training log saved to reports/training_br35h_baseline.log")


## 10. Checkpoint Verification
Verify that both `best_auc.pth` and `last_epoch.pth` exist in the save directory and have valid sizes.


In [ ]:
import os

CHECKPOINT_DIR = "./saved_models/e2ad_br35h/E2AD/0"
best_auc_path = os.path.join(CHECKPOINT_DIR, "best_auc.pth")
last_epoch_path = os.path.join(CHECKPOINT_DIR, "last_epoch.pth")

print(f"Checking checkpoints in: {CHECKPOINT_DIR}")
assert os.path.exists(CHECKPOINT_DIR), f"Checkpoint directory does not exist: {CHECKPOINT_DIR}"

for ckpt_name, ckpt_p in [("Best AUROC Model", best_auc_path), ("Final Epoch Model", last_epoch_path)]:
    if os.path.exists(ckpt_p):
        size_mb = os.path.getsize(ckpt_p) / (1024 * 1024)
        print(f"  [FOUND] {ckpt_name:<18} : {ckpt_p} ({size_mb:.2f} MB)")
    else:
        print(f"  [MISSING] {ckpt_name:<18} : {ckpt_p}")

assert os.path.exists(best_auc_path), "best_auc.pth was NOT saved!"
assert os.path.exists(last_epoch_path), "last_epoch.pth was NOT saved!"
print("Checkpoint verification SUCCESSFUL.")


## 11. Results Extraction & Metric Analysis
Parse the training logs and report the peak performance achieved on BR35H.


In [ ]:
import re
import numpy as np

best_auc_logged = 0.0
best_iter_logged = 0
final_auc_logged = 0.0

iter_regex = re.compile(r"(\d+)\s+iteration,\s+({.*}),\s+BEST_EVAL_AUC:\s+([0-9.]+),\s+at\s+(\d+)\s+iters")
last_eval_dict = {}

with open("reports/training_br35h_baseline.log", "r", encoding="utf-8") as f:
    for line in f:
        m = iter_regex.search(line)
        if m:
            it_num = int(m.group(1))
            best_auc_logged = float(m.group(3))
            best_iter_logged = int(m.group(4))
            try:
                eval_str = m.group(2).replace("array(", "").replace(")", "")
                last_eval_dict = eval(eval_str)
            except Exception:
                pass

print("=" * 65)
print("EXTRACTED BR35H BASELINE METRICS")
print("=" * 65)
print(f"Best Iteration       : {best_iter_logged} / 4000")
print(f"Best AUROC (logged)  : {best_auc_logged * 100:.2f}%")
if 'eval/f1' in last_eval_dict:
    print(f"F1 Score (final)     : {last_eval_dict['eval/f1'] * 100:.2f}%")
if 'eval/acc' in last_eval_dict:
    print(f"Accuracy (final)     : {last_eval_dict['eval/acc'] * 100:.2f}%")
if 'eval/recall' in last_eval_dict:
    print(f"Sensitivity (final)  : {last_eval_dict['eval/recall'] * 100:.2f}%")
if 'eval/specificity' in last_eval_dict:
    print(f"Specificity (final)  : {last_eval_dict['eval/specificity'] * 100:.2f}%")
print("=" * 65)


## 12. Reproducibility & Experiment Report

### Comparison against IEEE TMI 2025 Paper (Table IV - BR35H)

| Metric | Official Paper (IEEE TMI 2025) | Our Kaggle Baseline (`seed=0`) | Status |
| :--- | :--- | :--- | :--- |
| **AUROC (%)** | **99.83%** | *(Extracted in Section 11)* | Verified on GPU Run |
| **F1-Score (%)** | **99.62%** | *(Extracted in Section 11)* | Verified on GPU Run |
| **Accuracy (%)** | **99.44%** | *(Extracted in Section 11)* | Verified on GPU Run |
| **Sensitivity (%)** | *Not explicitly in Table IV* | *(Extracted in Section 11)* | Computed |
| **Specificity (%)** | *Not explicitly in Table IV* | *(Extracted in Section 11)* | Computed |

### Experiment Metadata Summary
- **Experiment ID**: `E2AD-BR35H-OFFICIAL-BASELINE`
- **Model Architecture**: ResNet-50 Encoder + Spatial Attention (SASC) + Task-specific CL (TCCL) + Dual Decoders
- **Training Iterations**: 4,000 iterations (Batch Size 32, Eval Frequency 100)
- **Optimizer**: AdamW ($\text{lr}=5 \times 10^{-4}$, $\text{lr}_{\text{encoder}}=5 \times 10^{-5}$, $\text{weight\_decay}=10^{-4}$)
- **Loss Formulation**: $\mathcal{L} = \lambda_1 \mathcal{L}_{\text{rec}} + \lambda_2 \mathcal{L}_{\text{cos}} + \lambda_3 \mathcal{L}_{\text{TCCL}} + \lambda_{\text{align}} \mathcal{L}_{\text{align}}$
- **Codebase Source**: [gnanadeep256/E2AD](https://github.com/gnanadeep256/E2AD)
- **Execution Target**: Kaggle GPU (Tesla T4, 16GB VRAM, PyTorch 2.x, CUDA 12.x)
- **Checkpoints Saved**: `saved_models/e2ad_br35h/E2AD/0/best_auc.pth`, `last_epoch.pth`
